In [36]:
import os
import json
import datetime
from dotenv import load_dotenv
from typing import TypedDict, List, Annotated, Dict, Literal
from datetime import datetime
from pydantic import BaseModel, Field
from operator import add
import shlex

from pathlib import Path
import base64
from icalendar import Calendar, Event
import pytz
from reader import Reader

from langchain_core.messages import HumanMessage, BaseMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI

from logger import Logger

from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

In [37]:
class TimeResult(BaseModel):
    """Схема ответа от Time агента"""
    schedule: dict[str, str] = Field(
        description="Расписание в формате {'год:месяц:число:час:минута-год:месяц:число:час:минута': 'событие'}, если чего то не хватает, пытайся вытащить по контексту, но все значения должны быть, дата сейчас: " + str(datetime.now())
    )

class ValidationResult(BaseModel):
    """Схема ответа от Validation агента"""
    next: Literal['Time', 'END'] = Field(
        description="Решение: Time - доработать, END - все в норме"
    )
    reason: str = Field(
        description="Краткое обоснование решения"
    )

In [38]:
#--------------------------MODELS CREATION------------------------------
load_dotenv('api_keys.env')

GITHUB_PAT = os.getenv('GITHUB_PAT')
GEMINI_API_KEY1 = os.getenv('GEMINI_API_KEY1')
GEMINI_API_KEY2 = os.getenv('GEMINI_API_KEY2')

GITHUB_URL = os.getenv('GITHUB_URL')

GPT_MODEL = os.getenv('GPT_MODEL')
GEMINI_MODEL = os.getenv('GEMINI_MODEL')

main_model = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    api_key=GEMINI_API_KEY1,
    temperature=0.1,
    max_retries=2
)

sub_model = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    api_key=GEMINI_API_KEY2,
    max_retries=2
)

# sub_model = ChatOpenAI(
#     model=GPT_MODEL,
#     api_key=GITHUB_PAT,
#     base_url=GITHUB_URL,
#     max_retries=5
# )

#------------------------------------------GRAPH LOGIC---------------------------------
class GraphState(TypedDict):
    schedule: dict[str, str]
    
    user_task: str
    val_query: str
    next_agent: str
    image: str
    
    content: str
    attempts: int
    
Logger.init_log_file('logs.txt')

#------------------------------------------LOADING PROMPTS----------------------------
def open_prompt(file_path: str = 'prompts.json') -> dict:
    with open(file_path, 'r') as f:
        return json.load(f)
    
PROMPTS = open_prompt()
    
#--------------------------------------CREATIN NODES-------------------------------
def Time(state: GraphState) -> GraphState:
    Logger.write_log('Time manager', 'вход')
    
    prompt = json.dumps(PROMPTS['Time'], ensure_ascii=False, indent=2)
    user_query = state['user_task']
    val_query = state.get('val_query', '')

    last_schedule = state.get('schedule', '')
    if last_schedule:
        last_schedule = "Добавь в это расписание все по запросу пользователя " + str(last_schedule)
    
    structured_model = main_model.with_structured_output(TimeResult)
    response = structured_model.invoke([HumanMessage([prompt, val_query, user_query, last_schedule])])
    
    schedule = response.schedule
    schedule = list(schedule.items())
    schedule.sort()
        
    Logger.write_log('Time manager', 'Создано расписание', str(schedule))
    
    return {
        'schedule': dict(schedule)
    }
    
def IMG(state: GraphState) -> GraphState:
    Logger.write_log('Image analyzer', 'вход')
    
    prompt = json.dumps(PROMPTS['IMG'], ensure_ascii=False, indent=2)
    user_query = state['image']
    message = HumanMessage(content=[
            {"type": "text", "text": prompt},
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{user_query}"}
            }
        ])
    
    response = main_model.invoke([message])
    
    try:
        res = response.content[0]['text']
        
    except:
        res = response.content
        Logger.write_log('Image analyzer', 'Внимание!! Возможно, вы используете не предпологаемую кодом модель!!', 'Предпологается gemini-3.1-flash-lite')
    
    Logger.write_log('Image analyzer', 'Изображение проанализировано', res)
    
    return {
        'user_task': state['user_task'] + res
    }
    
def VAL(state: GraphState) -> GraphState:
    Logger.write_log('Validation agent', 'вход')
    
    prompt = json.dumps(PROMPTS['VAL'], ensure_ascii=False, indent=2)
    user_query = state['user_task']
    schedule = state['schedule']
    structured_model = sub_model.with_structured_output(ValidationResult)
    
    response = structured_model.invoke([
        {"role": "system", "content": prompt},
        {"role": "user", "content": f"Расписание: {schedule}\nЗапрос пользователя: {user_query}"}
    ])
    
    next = response.next
    task = '' if next == 'END' else response.reason
    
    Logger.write_log('Validation agent', f'Передаю {next}', task)
    
    return {
        'val_query': task,
        'next_agent': next,
        'attempts': state.get('attempts', 0) + 1
    } 
    
def Visualizer(state: GraphState) -> GraphState:
    Logger.write_log('Visualizer', 'вход')
    
    prompt = json.dumps(PROMPTS['Visualizer'], ensure_ascii=False, indent=2)
    user_query = state['user_task']
    schedule = state['schedule']
    
    formatted_schedule = {}
    for key, value in schedule.items():
        # key = '2023:10:02:09:00'
        parts = key.split(':')
        if len(parts) == 5:
            year, month, day, hour, minute = parts
            # Создаем читаемый ключ
            date_obj = datetime.date(int(year), int(month), int(day))
            weekday = date_obj.strftime('%A').upper()
            month_name = date_obj.strftime('%B').upper()
            readable_key = f"{weekday}, {int(day)} {month_name} {hour}:{minute}"
            formatted_schedule[readable_key] = value
        else:
            formatted_schedule[key] = value
    
    response = sub_model.invoke([
        {"role": "system", "content": prompt},
        {"role": "user", "content": f"Расписание: {json.dumps(formatted_schedule, ensure_ascii=False)}\nЗапрос: {user_query}"}
    ])
    
    try:
        res = response.content[0]['text']
        Logger.write_log('Visualizer', 'Внимание!! Возможно, вы используете не предпологаемую кодом модель!!', 'Предпологается gpt-4.1-mini')
        
    except:
        res = response.content
    
    Logger.write_log('Visualizer', "Готово", res)
    
    return {
        'content': res
    }

#-----------------------------------CONDITIONAL EDGES LOGIC------------------------------
def ENTER_router(state: GraphState) -> str:
    img = state.get('image', None)
    if img:
        return 'IMG'
    return 'Time'
    
def VAL_router(state: GraphState) -> str:
    if state.get('attempts', 0) < 2: return state['next_agent']
    return 'END'
    
#-------------------------------------------GRAPH CREATING-------------------------------------
memo = InMemorySaver()
workflow = StateGraph(GraphState)

workflow.add_node('IMG', IMG)
workflow.add_node('Time', Time)
workflow.add_node('VAL', VAL)
workflow.add_node('Visualizer', Visualizer)

workflow.set_conditional_entry_point(
    ENTER_router,
    {'IMG': 'IMG', 'Time': 'Time'}
)

workflow.add_edge('IMG', 'Time')
workflow.add_edge('Time', 'VAL')

workflow.add_conditional_edges(
    'VAL',
    VAL_router,
    {'Time': 'Time', 'END': 'Visualizer'}
)

workflow.set_finish_point('Visualizer')

app = workflow.compile(checkpointer=memo)
config = {"configurable": {"thread_id": "session_1"}}


In [ ]:
def make_base_64(file_path: str) -> str:
    with open(file_path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')
    
def create_ics_from_dict(events_dict: dict, filename: str = 'my_calendar.ics', zone: str = "Europe/Moscow"):
    cal = Calendar()
    cal.add('prodid', '-//My calendar//mxm.dk//')
    cal.add('version', '2.0')
    
    selected_timezone = pytz.timezone(zone)
    print(f"\nВыбран часовой пояс: {zone}")
    
    for time in events_dict:
        start, end = time.split('-')
        s_year, s_moth, s_day, s_hour, s_minute = map(int, start.split(':'))
        e_year, e_moth, e_day, e_hour, e_minute = map(int, end.split(':'))
        summary = events_dict[time]
        
        start = selected_timezone.localize(datetime(s_year, s_moth, s_day, s_hour, s_minute))
        end = selected_timezone.localize(datetime(e_year, e_moth, e_day, e_hour, e_minute))
        
        event = Event()
        event.add('summary', summary)
        event.add('dtstart', start)
        event.add('dtend', end)
        
        cal.add_component(event)
        
    with open(filename, 'wb') as f:
        f.write(cal.to_ical())
    
    print(f'📁 Файл создан: {os.path.abspath(filename)}')
    print('Перейдите по ссылке и импортируйте его: https://calendar.google.com/calendar/u/0/r/settings/export?pli=1')

image_extensions = ['.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff', '.tif', '.avif', '.heic', '.jp2', '.dng']

while True:
    data = input("Введите сюда расписание, задачу и путь к файлу.")
    if not data:
        break
    
    try:
        parts = shlex.split(data) 
    except ValueError:
        parts = data.split()

    text_parts = []
    images = []
    for part in parts:
        clean_part = part.strip('"\'')
        if any(clean_part.lower().endswith(ext) for ext in image_extensions) and os.path.exists(clean_part):
            images.append(make_base_64(clean_part))
        elif Reader.is_supported(clean_part) and os.path.exists(clean_part):
            text_parts.append(Reader.read_document(clean_part))
        else:
            text_parts.append(part)
            
    user_input = ' '.join(text_parts)
    images = ' '.join(images)
            
    if images:
        response = app.invoke({
            'user_task': user_input,
            'image': images
        }, config=config)
        
    else:
        response = app.invoke({
            'user_task': user_input
        }, config=config)
        
    print(response['content'])
    
ans = input("Хотите получить рассылку на данный календарь? [Да/Нет]")
if not ans or ans.lower() == 'да':
    create_ics_from_dict(response['schedule'])

/home/lzera/Рабочий стол/Projects/PROJ_SUMMER/.venv/lib/python3.12/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


{'time': '2026-06-18 00:49:23', 'agent_name': 'Time manager', 'action': 'вход', 'dop_info': ''}
{'time': '2026-06-18 00:49:26', 'agent_name': 'Time manager', 'action': 'Создано расписание', 'dop_info': "[('2026:06:22:07:30-2026:06:22:08:00', 'Завтрак'), ('2026:06:22:08:00-2026:08:30:08:30', 'Бизнес: ле..."}
{'time': '2026-06-18 00:49:26', 'agent_name': 'Validation agent', 'action': 'вход', 'dop_info': ''}
{'time': '2026-06-18 00:49:27', 'agent_name': 'Validation agent', 'action': 'Передаю Time', 'dop_info': "В расписании обнаружена ошибка в записи '2026:06:22:08:00-2026:08:30:08:30', где дата окончания указ..."}
{'time': '2026-06-18 00:49:27', 'agent_name': 'Time manager', 'action': 'вход', 'dop_info': ''}
{'time': '2026-06-18 00:49:30', 'agent_name': 'Time manager', 'action': 'Создано расписание', 'dop_info': "[('2026:06:22:07:30-2026:06:22:08:00', 'Завтрак'), ('2026:06:22:08:00-2026:06:22:08:30', 'Бизнес: ле..."}
{'time': '2026-06-18 00:49:30', 'agent_name': 'Validation agent', 'acti